# ProxyTrace 02. 문서 정리

`01_fetch_proxy.ipynb`에서 받은 위임장 HTML을 **쪽 번호가 붙은 깨끗한 텍스트**로 바꿔요.

| 처리 | 이유 |
| --- | --- |
| 숨김 영역 제거 | SEC 문서 앞부분에 화면에 안 보이는 XBRL 정보가 들어 있어요 |
| 페이지 나누기 | `page-break` 스타일을 기준으로 페이지를 나눠요 |
| 머리말·꼬리말 제거 | 여러 페이지에 반복되는 문구(예: `2026 Proxy Statement \| 9`)를 자동으로 찾아서 지워요 |
| 쪽 번호 기록 | 꼬리말에서 쪽 번호를 읽어 두면 나중에 근거 위치를 "p.72"처럼 표시할 수 있어요 |

결과: `data/clean/{TICKER}/{제출일}_pages.json`, `{제출일}.txt`

01 노트북이 만든 `manifest.json`을 읽어서 처리하니까, **티커만 맞추면 어떤 기업이든 돌아가요.**

In [ ]:
!pip install beautifulsoup4 lxml -q

## 1. 설정

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

from bs4 import BeautifulSoup

# ▶ 01 노트북과 같은 티커
TICKER = "META"

RAW_DIR = Path("data/raw") / TICKER.lower()
CLEAN_DIR = Path("data/clean") / TICKER.lower()
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

manifest = json.loads((RAW_DIR / "manifest.json").read_text(encoding="utf-8"))
PROXIES = manifest["proxies"]          # [최신, 직전]
print(manifest["company"], [p["filingDate"] for p in PROXIES])

PAGE_MARK = "<<<PAGE_BREAK>>>"

## 2. 함수 정의

In [ ]:
BLOCK_TAGS = ["p", "div", "tr", "li", "h1", "h2", "h3", "h4", "h5", "h6", "table"]
CELL_TAGS = ["td", "th"]


def remove_hidden(soup: BeautifulSoup) -> int:
    """화면에 안 보이는 영역 제거: script/style, XBRL 머리 정보, display:none"""
    n = 0
    for t in soup(["script", "style", "head"]):
        t.decompose(); n += 1
    for t in soup.find_all(re.compile(r"^ix:header$", re.I)):
        t.decompose(); n += 1
    for t in soup.find_all(style=re.compile(r"display\s*:\s*none", re.I)):
        t.decompose(); n += 1
    return n


def mark_page_breaks(soup: BeautifulSoup) -> int:
    """page-break 스타일이 있는 곳에 페이지 구분 표시 넣기"""
    n = 0
    for el in soup.find_all(style=re.compile(r"page-break-(before|after)\s*:\s*always", re.I)):
        style = el.get("style", "").lower()
        if "page-break-before" in style:
            el.insert_before(PAGE_MARK)
        else:
            el.insert_after(PAGE_MARK)
        n += 1
    return n


def html_to_text(soup: BeautifulSoup) -> str:
    """블록 단위로 줄바꿈, 표 셀은 ' | '로 구분.
    SEC 문서는 글자 조각마다 <span>이 붙어 있어서 get_text("\n")을 쓰면 문장이 끊겨요."""
    for br in soup.find_all("br"):
        br.replace_with("\n")
    for t in soup.find_all(CELL_TAGS):
        t.append(" | ")
    for t in soup.find_all(BLOCK_TAGS):
        t.append("\n")
    text = soup.get_text("")
    text = text.replace("\xa0", " ").replace("​", "")
    lines = [re.sub(r"[ \t]+", " ", ln).strip(" |").strip() for ln in text.split("\n")]
    return "\n".join(ln for ln in lines if ln)


def edge_lines(page: str, k: int = 3) -> list[str]:
    lines = page.split("\n")
    return lines[:k] + lines[-k:]


def norm(line: str) -> str:
    """숫자를 #으로 바꿔서 '2026 Proxy Statement | 9'와 '| 10'을 같은 문구로 보기"""
    return re.sub(r"\d+", "#", line)


def find_repeated(pages: list[str], min_ratio: float = 0.3, max_len: int = 60) -> set[str]:
    """여러 페이지의 위·아래 끝에 반복되는 '짧은' 문구 = 머리말·꼬리말
    - min_ratio: 전체 페이지 중 이 비율 이상에서 반복되면 머리말·꼬리말로 판단
    - max_len: 이보다 긴 줄은 본문 문장일 가능성이 높아서 제외"""
    cnt = Counter()
    for p in pages:
        cnt.update({norm(ln) for ln in edge_lines(p) if len(ln) <= max_len})
    th = max(3, int(len(pages) * min_ratio))
    return {line for line, c in cnt.items() if c >= th}


def page_number(page: str, repeated: set[str]) -> int | None:
    """반복 문구 중 숫자가 들어 있는 줄에서 쪽 번호 읽기"""
    for ln in reversed(edge_lines(page)):
        if norm(ln) in repeated and "#" in norm(ln):
            nums = re.findall(r"\d+", ln)
            # '2026 Proxy Statement | 9'처럼 연도가 같이 있으면 마지막 숫자가 쪽 번호
            nums = [int(x) for x in nums if not (1900 <= int(x) <= 2100)]
            if nums:
                return nums[-1]
    return None


def strip_repeated(page: str, repeated: set[str]) -> str:
    lines = page.split("\n")
    edge_idx = set(range(min(3, len(lines)))) | set(range(max(0, len(lines) - 3), len(lines)))
    keep = [ln for i, ln in enumerate(lines) if not (i in edge_idx and norm(ln) in repeated)]
    return "\n".join(keep)


def clean_document(html: str) -> dict:
    soup = BeautifulSoup(html, "lxml")
    n_hidden = remove_hidden(soup)
    n_breaks = mark_page_breaks(soup)
    text = html_to_text(soup)

    raw_pages = [p.strip() for p in text.split(PAGE_MARK) if p.strip()]
    if len(raw_pages) <= 1:
        print("⚠ 페이지 구분 표시를 못 찾았어요. 문서 전체를 한 페이지로 처리합니다.")

    repeated = find_repeated(raw_pages) if len(raw_pages) > 3 else set()
    pages = []
    for i, p in enumerate(raw_pages):
        pages.append({
            "page_idx": i,                       # 문서 안 순서 (0부터)
            "page_no": page_number(p, repeated), # 인쇄된 쪽 번호 (없으면 None)
            "text": strip_repeated(p, repeated),
        })
    return {
        "stats": {"hidden_removed": n_hidden, "page_breaks": n_breaks,
                  "pages": len(pages), "repeated_lines": sorted(repeated)},
        "pages": pages,
    }

## 3. 최신 위임장 먼저 처리해 보기

In [ ]:
html = Path(PROXIES[0]["saved_to"]).read_text(encoding="utf-8", errors="ignore")
doc = clean_document(html)

s = doc["stats"]
print(f"숨김 요소 제거: {s['hidden_removed']}개")
print(f"페이지 구분 표시: {s['page_breaks']}개 → 페이지 {s['pages']}개")
print("반복 문구(머리말·꼬리말로 판단):")
for line in s["repeated_lines"]:
    print("  ", line)

## 4. 결과 눈으로 확인하기 ⚠ 중요
아래를 꼭 확인하세요. 문제가 있으면 함수를 고쳐야 해요.

- **반복 문구 목록**에 본문 문장이 섞여 있지 않은지 (섞여 있으면 `min_ratio`를 올리기)
- **쪽 번호**가 1, 2, 3… 순서대로 나오는지 (표지·목차는 `None`이어도 괜찮아요)
- 주주제안 부분이 제대로 잡히는지 (아래 `find()` 결과)

In [ ]:
# 쪽 번호가 순서대로 나오는지
[(p["page_idx"], p["page_no"]) for p in doc["pages"]][:30]

In [ ]:
# 특정 쪽 내용 보기 (주주제안 시작 페이지)
def show(page_no: int, n_chars: int = 1500):
    for p in doc["pages"]:
        if p["page_no"] == page_no:
            print(f"--- p.{page_no} (idx {p['page_idx']}) ---")
            print(p["text"][:n_chars])
            return
    print(f"p.{page_no}를 못 찾았어요")

# 주주제안이 처음 나오는 쪽을 자동으로 찾아서 보여줘요
first = next((p for p in doc["pages"] if re.search(r"shareholder proposal|stockholder proposal", p["text"], re.I)
              and p["page_idx"] > len(doc["pages"]) * 0.3), None)
if first: show(first["page_no"]) if first["page_no"] else print(first["text"][:1500])
else: print("주주제안 부분을 못 찾았어요")

In [ ]:
# 키워드가 몇 쪽에 있는지 찾기 → 02 다음 단계(안건 추출) 규칙 만들 때 써요
def find(keyword: str):
    hits = [(p["page_no"], p["page_idx"]) for p in doc["pages"] if keyword.lower() in p["text"].lower()]
    print(f"'{keyword}': {hits}")

# 회사마다 문구가 달라서 후보를 여러 개 넣어 봐요.
# 여기서 많이 잡히는 문구가 03 노트북(안건 추출)의 경계 규칙 후보가 돼요.
CANDIDATES = [
    "Shareholder Proposal", "Stockholder Proposal",
    "Statement in Opposition", "Board's Statement", "Board of Directors' Statement",
    "Our Board's Response", "Company's Response",
    "recommends a vote AGAINST", "Supporting Statement",
]
for kw in CANDIDATES:
    find(kw)

## 5. 최신·직전 위임장 모두 저장

In [ ]:
for p in PROXIES:
    src = Path(p["saved_to"])
    d = clean_document(src.read_text(encoding="utf-8", errors="ignore"))
    d["meta"] = {k: p[k] for k in ("filingDate", "accessionNumber", "role", "url")}
    stem = p["filingDate"]

    (CLEAN_DIR / f"{stem}_pages.json").write_text(
        json.dumps(d, ensure_ascii=False, indent=2), encoding="utf-8")

    # 사람이 읽기 좋은 전체 텍스트 (쪽 표시 포함)
    full = "\n\n".join(f"===== p.{p['page_no']} (idx {p['page_idx']}) =====\n{p['text']}" for p in d["pages"])
    (CLEAN_DIR / f"{stem}.txt").write_text(full, encoding="utf-8")

    print(f"[{p['role']}] {stem}: 페이지 {d['stats']['pages']}개 저장")

## 다음 단계
4번 셀의 `find()` 결과를 보고 알려 주세요. 가능하면 **2~3개 기업**으로 돌려 보세요 (예: META, WMT, MSFT).
기업마다 어떤 문구가 잡히는지 비교하면, 03 노트북(안건 추출)에서 **후보 문구 목록 + 실패 시 LLM 판단** 규칙을 만들 수 있어요.